# Facial Emotion Recognition — FER2013 (Max Accuracy)

Transfer-learning based facial emotion classifier trained on **FER2013**, extending a
DCNN baseline paper (65.68% val accuracy) using **EfficientNetV2M**, **MixUp**,
**Focal Loss**, a 3-phase progressive fine-tuning schedule, and **10-crop Test-Time
Augmentation (TTA)** — reaching **68.42% TTA test accuracy**.

> See the accompanying `FER_Presentation.pptx` for the full project write-up, architecture
> diagrams, and results comparison against the base paper and other backbones.

---


## 1. Setup & Configuration

Imports TensorFlow/Keras, NumPy, and scikit-learn utilities, seeds all RNGs for
reproducibility, and detects available GPUs.

In [ ]:
# ================================================================
#  FACIAL EMOTION RECOGNITION — MAXIMUM ACCURACY VERSION
#  Target: 73-76% test accuracy (well above paper's 65.68%)
#
#  ROOT CAUSE ANALYSIS of previous 63.9% result:
#   - Cosine LR decayed too fast → model under-trained
#   - Phase 2 best val was only ~52%, still improving
#   - Backbone too small (EfficientNetV2S)
#   - No MixUp/CutMix augmentation
#   - No multi-crop TTA
#
#  FIXES IN THIS VERSION:
#   1. Switch to EfficientNetV2M (larger backbone)
#   2. Fix LR schedule — warm restarts, not cosine decay
#   3. MixUp augmentation (huge for FER)
#   4. More epochs with proper patience
#   5. AdamW optimizer with weight decay
#   6. 5-crop TTA (center + 4 corners)
#   7. Larger image: 128x128
#   8. Focal loss to handle class imbalance better
#
#  SETUP: GPU T4 x2 recommended (or P100)
# ================================================================

import os, sys, random, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import EfficientNetV2M
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("=" * 55)
print(f"  TensorFlow : {tf.__version__}")
print(f"  GPUs       : {len(gpus)}")
for g in gpus: print(f"    {g.name}")
if not gpus: print("  WARNING: No GPU — enable it!")
print("=" * 55)

## 2. Auto-Detect Dataset

Scans the input directory for either pre-split image folders (`train/`, `test/`) or a
raw FER2013 CSV file, and picks the loading strategy accordingly.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Auto-detect Dataset                               ║
# ╚══════════════════════════════════════════════════════════════╝
print("\nScanning /kaggle/input ...")

TRAIN_DIR = None; TEST_DIR = None; CSV_PATH = None

for root, dirs, files in os.walk('/kaggle/input'):
    level  = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level <= 2:
        sub = '  ' * (level + 1)
        for f in files[:3]: print(f"{sub}{f}")
        if len(files) > 3: print(f"{sub}... +{len(files)-3} more")

for root, dirs, files in os.walk('/kaggle/input'):
    name    = os.path.basename(root).lower()
    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if name == 'train' and len(subdirs) >= 5:             TRAIN_DIR = root
    if name in ['test', 'validation', 'val'] and len(subdirs) >= 5: TEST_DIR = root
    for f in files:
        if f.endswith('.csv') and ('fer' in f.lower() or 'emotion' in f.lower()):
            CSV_PATH = os.path.join(root, f)

USE_CSV = False
if TRAIN_DIR and TEST_DIR:
    print(f"\nMode: IMAGE FOLDERS  | Train: {TRAIN_DIR} | Test: {TEST_DIR}")
elif CSV_PATH:
    print(f"\nMode: CSV  → {CSV_PATH}")
    USE_CSV = True
else:
    raise FileNotFoundError("Dataset not found! Add msambare/fer2013 to notebook.")

## 3. Config

Core hyperparameters: 128×128 RGB input, batch size 32, 7 emotion classes, and epoch
budgets for each of the 3 training phases.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Config                                            ║
# ╚══════════════════════════════════════════════════════════════╝
IMG_SIZE    = 128     # Larger = better for EfficientNetV2M
BATCH_SIZE  = 32
NUM_CLASSES = 7

# Training phases — longer phases, controlled by early stopping
EPOCHS_P1   = 20     # Head only (frozen backbone)
EPOCHS_P2   = 60     # Top-200 layers unfrozen
EPOCHS_P3   = 40     # Full backbone, very low LR

EMOTIONS     = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']
CSV_EMOTIONS = ['angry','disgust','fear','happy','sad','surprise','neutral']

OUT_DIR = '/kaggle/working'
os.makedirs(f'{OUT_DIR}/checkpoints', exist_ok=True)

print(f"  Backbone    : EfficientNetV2M (larger than V2S!)")
print(f"  Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Phase 1 epochs: {EPOCHS_P1}")
print(f"  Phase 2 epochs: {EPOCHS_P2}")
print(f"  Phase 3 epochs: {EPOCHS_P3}")

## 4. CSV → Images

If the dataset was provided as the classic FER2013 CSV (48×48 grayscale pixel strings),
converts every row into a resized RGB image file organized into class folders.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — CSV → Images (skip if folder mode)               ║
# ╚══════════════════════════════════════════════════════════════╝
if USE_CSV:
    print("Converting CSV to image files...")
    df = pd.read_csv(CSV_PATH)
    df.columns = [c.lower().strip() for c in df.columns]
    label_col = 'emotion'; pixel_col = 'pixels'
    usage_col = next((c for c in ['usage','split','set'] if c in df.columns), None)
    for idx, row in df.iterrows():
        eid   = int(row[label_col])
        ename = CSV_EMOTIONS[eid]
        px    = np.array(row[pixel_col].split(), dtype=np.uint8).reshape(48, 48)
        img   = Image.fromarray(px, 'L').convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        split = ('train' if str(row[usage_col]).lower() in ['training','train'] else 'test') if usage_col else ('train' if idx < int(len(df)*0.85) else 'test')
        folder = f'{OUT_DIR}/images/{split}/{ename}'
        os.makedirs(folder, exist_ok=True)
        img.save(f'{folder}/{idx}.png')
        if (idx+1) % 5000 == 0: print(f"  {idx+1}/{len(df)} done...")
    TRAIN_DIR = f'{OUT_DIR}/images/train'
    TEST_DIR  = f'{OUT_DIR}/images/test'
    print("Done!")

print("\nDataset counts:")
for split, d in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    if d and os.path.exists(d):
        classes = sorted([c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c))])
        total   = sum(len(os.listdir(os.path.join(d, c))) for c in classes)
        print(f"  {split} ({total} total):")
        for c in classes:
            n = len(os.listdir(os.path.join(d, c)))
            print(f"    {c:12s}: {n:5d}  {'█'*(n//200)}")

## 5. Data Generators + MixUp Setup

Builds Keras `ImageDataGenerator`s with heavy augmentation (flips, rotation, shifts,
zoom, brightness, channel shift) for training, and lighter generators for
validation/test.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Data Generators + MixUp                          ║
# ╚══════════════════════════════════════════════════════════════╝
# KEY FIX: NO rescale anywhere — EfficientNetV2M handles it internally
# IMPROVEMENT: MixUp augmentation layer (huge for FER)

train_datagen = ImageDataGenerator(
    validation_split    = 0.15,
    horizontal_flip     = True,
    rotation_range      = 20,
    width_shift_range   = 0.12,
    height_shift_range  = 0.12,
    zoom_range          = 0.20,
    shear_range         = 0.10,
    brightness_range    = [0.65, 1.35],
    channel_shift_range = 15.0,
    fill_mode           = 'nearest',
)

val_datagen  = ImageDataGenerator(validation_split=0.15)
test_datagen = ImageDataGenerator()

flow_args = dict(
    target_size = (IMG_SIZE, IMG_SIZE),
    color_mode  = 'rgb',
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    seed        = SEED,
)

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, subset='training',   shuffle=True,  **flow_args)
val_gen   = val_datagen.flow_from_directory(  TRAIN_DIR, subset='validation', shuffle=False, **flow_args)
test_gen  = test_datagen.flow_from_directory( TEST_DIR,  shuffle=False,                      **flow_args)

idx_to_name = {v: k for k, v in train_gen.class_indices.items()}
print(f"Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")
print(f"Classes: {train_gen.class_indices}")

## 6. MixUp Generator Wrapper

Wraps the training generator to blend pairs of images/labels (`x = λ·x₁ + (1-λ)·x₂`),
a regularizer shown to boost FER accuracy by ~2–3%. Also computes balanced class
weights to offset the under-represented `Disgust` class.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — MixUp Generator Wrapper                          ║
# ╚══════════════════════════════════════════════════════════════╝
# MixUp blends two training samples: x = λ*x1 + (1-λ)*x2
# This acts as a strong regularizer and boosts FER accuracy ~2-3%

def mixup_generator(gen, alpha=0.3, class_weights=None):
    """Wraps a Keras generator and applies MixUp augmentation.
    Yields (x, y, sample_weight) so class_weight works with generators."""
    cw_tensor = None
    if class_weights is not None:
        cw_tensor = np.array([class_weights[i] for i in range(len(class_weights))])

    while True:
        x1, y1 = next(gen)
        x2, y2 = next(gen)
        bsz = min(len(x1), len(x2))
        x1, y1, x2, y2 = x1[:bsz], y1[:bsz], x2[:bsz], y2[:bsz]
        lam = np.random.beta(alpha, alpha)
        x_mix = lam * x1 + (1 - lam) * x2
        y_mix = lam * y1 + (1 - lam) * y2

        if cw_tensor is not None:
            # Compute per-sample weight as weighted average of the two mixed classes
            sw = lam * (y1 @ cw_tensor) + (1 - lam) * (y2 @ cw_tensor)
            yield x_mix.astype(np.float32), y_mix.astype(np.float32), sw.astype(np.float32)
        else:
            yield x_mix.astype(np.float32), y_mix.astype(np.float32)

# Compute class weights (handles disgust class imbalance)
cw_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes,
)
class_weight_dict = dict(enumerate(cw_array))
print("Class weights:")
for i, w in class_weight_dict.items():
    print(f"  {idx_to_name.get(i,'?'):12s}: {w:.3f}")

## 7. Focal Loss

Custom loss function that down-weights easy/confident examples and focuses training
signal on hard ones — more effective than plain cross-entropy on FER2013's class
imbalance.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Focal Loss (handles class imbalance better)      ║
# ╚══════════════════════════════════════════════════════════════╝
# Focal Loss down-weights easy examples, focuses on hard ones.
# Better than plain cross-entropy for imbalanced datasets like FER.

class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.ls    = label_smoothing

    def call(self, y_true, y_pred):
        # Apply label smoothing
        n = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true = y_true * (1 - self.ls) + self.ls / n

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce     = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))

print("Focal Loss defined (gamma=2.0, label_smoothing=0.1)")

## 8. Model — EfficientNetV2M + Custom Head

Builds the classifier: an ImageNet-pretrained **EfficientNetV2M** backbone (~54M
params) feeding a dual global-pooling head (avg + max), followed by two dense blocks
with batch norm, Swish activation, and dropout, trained with **AdamW**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Model: EfficientNetV2M + Improved Head           ║
# ╚══════════════════════════════════════════════════════════════╝
# KEY UPGRADE: EfficientNetV2M has 54M params vs V2S's 21M
# Much stronger feature extractor for emotion recognition

def build_model(unfreeze_top=0, lr=1e-3, use_focal=True):
    base = EfficientNetV2M(
        include_top           = False,
        weights               = 'imagenet',
        input_shape           = (IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing = True,  # expects raw [0,255]
    )
    base.trainable = False

    if unfreeze_top == 999:  # Unfreeze all
        base.trainable = True
    elif unfreeze_top > 0:
        for layer in base.layers[:-unfreeze_top]:
            layer.trainable = False
        for layer in base.layers[-unfreeze_top:]:
            layer.trainable = True

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=(unfreeze_top > 0))

    # Dual-pooling head
    avg = layers.GlobalAveragePooling2D()(x)
    mx  = layers.GlobalMaxPooling2D()(x)
    x   = layers.Concatenate()([avg, mx])

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.45)(x)

    # Dense block 1
    x = layers.Dense(768, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Dropout(0.35)(x)

    # Dense block 2
    x = layers.Dense(384, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.Dropout(0.25)(x)

    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inp, out)

    loss_fn = FocalLoss(gamma=2.0, label_smoothing=0.1) if use_focal else \
              keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    # AdamW: Adam + proper weight decay (better than L2 reg alone)
    optimizer = keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    return model


model = build_model(unfreeze_top=0, lr=1e-3)
total_p   = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Backbone        : EfficientNetV2M")
print(f"Total params    : {total_p:,}")
print(f"Trainable params: {trainable:,}  (backbone frozen)")

## 9. Smart Callbacks — Warm Restarts

Defines a *flat-then-cosine* learning-rate schedule (linear warmup → flat plateau →
cosine decay) plus checkpointing, early stopping, and CSV logging — fixing the
previous version's too-aggressive pure cosine decay.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Smart Callbacks with Warm Restarts               ║
# ╚══════════════════════════════════════════════════════════════╝
# KEY FIX: Previous version used cosine decay that hit ~0 too fast
# Now using: flat LR for first half, cosine for second half
# This keeps LR high enough to learn, then smoothly reduces it

def make_callbacks(phase, total_epochs, initial_lr, min_lr=1e-7, patience_es=15):

    def flat_then_cosine(epoch):
        """Stay at initial_lr for first 30% of epochs, then cosine decay."""
        warmup_epochs = max(2, int(total_epochs * 0.10))
        flat_epochs   = int(total_epochs * 0.30)

        if epoch < warmup_epochs:
            # Linear warmup
            return float(min_lr + (initial_lr - min_lr) * epoch / warmup_epochs)
        elif epoch < flat_epochs:
            # Flat phase
            return float(initial_lr)
        else:
            # Cosine decay
            progress = (epoch - flat_epochs) / max(1, total_epochs - flat_epochs)
            cosine   = 0.5 * (1 + np.cos(np.pi * progress))
            return float(min_lr + (initial_lr - min_lr) * cosine)

    return [
        callbacks.ModelCheckpoint(
            filepath          = f'{OUT_DIR}/checkpoints/best_p{phase}.weights.h5',
            monitor           = 'val_accuracy',
            save_best_only    = True,
            save_weights_only = True,
            mode              = 'max',
            verbose           = 1,
        ),
        callbacks.EarlyStopping(
            monitor              = 'val_accuracy',
            patience             = patience_es,
            restore_best_weights = True,
            verbose              = 1,
        ),
        callbacks.LearningRateScheduler(flat_then_cosine, verbose=0),
        callbacks.CSVLogger(f'{OUT_DIR}/log_p{phase}.csv'),
    ]

## 10. Phase 1 — Train Head Only

Trains just the classification head with the backbone fully frozen (LR 1e-3, up to 20
epochs) to warm up the new layers before touching the pretrained weights.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Phase 1: Train Head Only (Frozen Backbone)      ║
# ╚══════════════════════════════════════════════════════════════╝
print("=" * 60)
print("  PHASE 1 — Head only (backbone frozen)")
print(f"  LR: 1e-3, Epochs: up to {EPOCHS_P1}")
print("=" * 60)

mixup_train = mixup_generator(train_gen, alpha=0.3, class_weights=class_weight_dict)
steps_per_epoch = train_gen.samples // BATCH_SIZE

history_p1 = model.fit(
    mixup_train,
    steps_per_epoch = steps_per_epoch,
    epochs          = EPOCHS_P1,
    validation_data = val_gen,
    callbacks       = make_callbacks(phase=1, total_epochs=EPOCHS_P1,
                                     initial_lr=1e-3, patience_es=8),
    verbose         = 1,
)

best_p1 = max(history_p1.history['val_accuracy'])
print(f"\nPhase 1 best val accuracy: {best_p1*100:.2f}%")
model.save_weights(f'{OUT_DIR}/checkpoints/end_p1.weights.h5')

## 11. Phase 2 — Fine-Tune Top 200 Layers

Unfreezes the last 200 backbone layers and continues training at a lower LR (2e-4, up
to 60 epochs) from the best Phase 1 checkpoint.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Phase 2: Fine-tune Top 200 Layers               ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("  PHASE 2 — Fine-tuning top 200 backbone layers")
print(f"  LR: 2e-4 → cosine → 1e-7, Epochs: up to {EPOCHS_P2}")
print("=" * 60)

model2 = build_model(unfreeze_top=200, lr=2e-4)
model2.load_weights(f'{OUT_DIR}/checkpoints/end_p1.weights.h5')

trainable2 = sum(np.prod(v.shape) for v in model2.trainable_variables)
print(f"Trainable params (Phase 2): {trainable2:,}")

train_gen.reset()
mixup_train2 = mixup_generator(train_gen, alpha=0.3, class_weights=class_weight_dict)

history_p2 = model2.fit(
    mixup_train2,
    steps_per_epoch = steps_per_epoch,
    epochs          = EPOCHS_P2,
    validation_data = val_gen,
    callbacks       = make_callbacks(phase=2, total_epochs=EPOCHS_P2,
                                     initial_lr=2e-4, patience_es=15),
    verbose         = 1,
)

best_p2 = max(history_p2.history['val_accuracy'])
print(f"\nPhase 2 best val accuracy: {best_p2*100:.2f}%")
model2.save_weights(f'{OUT_DIR}/checkpoints/end_p2.weights.h5')

## 12. Phase 3 — Full Backbone Fine-Tune

Unfreezes the entire backbone and fine-tunes at a very low LR (5e-6, up to 40 epochs)
starting from the *best* (not last) Phase 2 checkpoint, with lighter augmentation to
avoid catastrophic forgetting.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Phase 3: Full Backbone Fine-tune                ║
# ╚══════════════════════════════════════════════════════════════╝
# CRITICAL: Start from the BEST P2 checkpoint (not end weights)
# Use very low LR (5e-6) with flat-then-cosine schedule

print("\n" + "=" * 60)
print("  PHASE 3 — Full backbone fine-tuning")
print(f"  LR: 5e-6 → cosine → 1e-8, Epochs: up to {EPOCHS_P3}")
print("=" * 60)

model3 = build_model(unfreeze_top=999, lr=5e-6)  # 999 = all layers

# Load BEST Phase 2 weights (not end_p2, which may be overfit)
best_p2_path = f'{OUT_DIR}/checkpoints/best_p2.weights.h5'
model3.load_weights(best_p2_path)
print(f"Loaded best Phase 2 weights from: {best_p2_path}")

trainable3 = sum(np.prod(v.shape) for v in model3.trainable_variables)
print(f"Trainable params (Phase 3): {trainable3:,}")

# Phase 3: lighter augmentation (model is close to convergence)
light_datagen = ImageDataGenerator(
    validation_split   = 0.15,
    horizontal_flip    = True,
    rotation_range     = 15,
    width_shift_range  = 0.10,
    height_shift_range = 0.10,
    zoom_range         = 0.12,
    brightness_range   = [0.8, 1.2],
    fill_mode          = 'nearest',
)
light_gen = light_datagen.flow_from_directory(
    TRAIN_DIR, subset='training', shuffle=True, **flow_args)

history_p3 = model3.fit(
    light_gen,
    epochs          = EPOCHS_P3,
    validation_data = val_gen,
    class_weight    = class_weight_dict,
    callbacks       = make_callbacks(phase=3, total_epochs=EPOCHS_P3,
                                     initial_lr=5e-6, min_lr=1e-8,
                                     patience_es=12),
    verbose         = 1,
)

best_p3 = max(history_p3.history['val_accuracy'])
print(f"\nPhase 3 best val accuracy: {best_p3*100:.2f}%")

## 13. Training Curves

Plots accuracy and loss across all three phases, with reference lines for the paper's
baseline and the 70% target.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Training Curves                                  ║
# ╚══════════════════════════════════════════════════════════════╝
acc   = history_p1.history['accuracy']     + history_p2.history['accuracy']     + history_p3.history['accuracy']
val   = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy'] + history_p3.history['val_accuracy']
loss  = history_p1.history['loss']         + history_p2.history['loss']         + history_p3.history['loss']
vloss = history_p1.history['val_loss']     + history_p2.history['val_loss']     + history_p3.history['val_loss']

eps    = range(1, len(acc)+1)
split1 = len(history_p1.history['accuracy'])
split2 = split1 + len(history_p2.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(eps, acc,  label='Train', color='royalblue', linewidth=2)
ax1.plot(eps, val,  label='Val',   color='tomato',    linewidth=2)
ax1.axvline(split1+0.5, linestyle='--', color='gray',   alpha=0.7, label='Phase 2')
ax1.axvline(split2+0.5, linestyle='--', color='purple', alpha=0.7, label='Phase 3')
ax1.axhline(0.6568, linestyle=':', color='orange', alpha=0.8, label='Paper 65.68%')
ax1.axhline(0.7000, linestyle=':', color='green',  alpha=0.8, label='Target 70%')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch')
ax1.set_ylim(0, 1); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(eps, loss,  label='Train', color='royalblue', linewidth=2)
ax2.plot(eps, vloss, label='Val',   color='tomato',    linewidth=2)
ax2.axvline(split1+0.5, linestyle='--', color='gray',   alpha=0.7)
ax2.axvline(split2+0.5, linestyle='--', color='purple', alpha=0.7)
ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f'Training — Best Val: {max(val)*100:.2f}%', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → training_curves.png")

## 14. Test Set Evaluation

Loads the best Phase 3 weights and evaluates straight (no TTA) accuracy on the held-out
test set.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Test Set Evaluation (straight predict)          ║
# ╚══════════════════════════════════════════════════════════════╝
best_p3_path = f'{OUT_DIR}/checkpoints/best_p3.weights.h5'
if os.path.exists(best_p3_path):
    model3.load_weights(best_p3_path)
    print("Loaded best Phase 3 weights")

test_gen.reset()
test_loss, test_acc = model3.evaluate(test_gen, verbose=1)
print(f"\nTest Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

## 15. Advanced TTA — 10-Crop Evaluation

Applies 5-crop (center + 4 corners) × horizontal-flip Test-Time Augmentation (10 views
per image) and averages predictions — the accuracy-boosting step that pushes the final
score above the paper baseline.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Advanced TTA (5-Crop + Horizontal Flip)         ║
# ╚══════════════════════════════════════════════════════════════╝
# UPGRADE: 5-crop TTA (center + 4 corners) × flip = 10 crops
# This is the gold standard for image classification TTA.
# Bug fixed: NO rescaling — raw [0,255] for EfficientNetV2M

print("\nRunning advanced 5-Crop TTA (10 views per image)...")

# Load test images into RAM (raw [0, 255])
raw_test = ImageDataGenerator()  # NO rescale!
raw_gen  = raw_test.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)

all_imgs, all_true = [], []
for imgs, lbls in raw_gen:
    all_imgs.append(imgs)
    all_true.append(lbls)
    if sum(len(x) for x in all_imgs) >= raw_gen.samples:
        break

all_imgs = np.vstack(all_imgs)[:raw_gen.samples]   # shape: (N, H, W, 3)
all_true = np.vstack(all_true)[:raw_gen.samples]
true_labels = np.argmax(all_true, axis=1)

print(f"  Loaded {all_imgs.shape[0]} images, range [{all_imgs.min():.0f}, {all_imgs.max():.0f}]")
assert all_imgs.max() > 1.0, "ERROR: Images appear to be rescaled! Check generator."


def five_crop(img, crop_size):
    """Returns 5 crops: center + 4 corners."""
    H, W = img.shape[:2]
    c    = (H - crop_size) // 2
    return [
        img[c:c+crop_size, c:c+crop_size],               # Center
        img[0:crop_size,   0:crop_size],                  # Top-left
        img[0:crop_size,   W-crop_size:W],                # Top-right
        img[H-crop_size:H, 0:crop_size],                  # Bottom-left
        img[H-crop_size:H, W-crop_size:W],                # Bottom-right
    ]


def resize_batch(imgs, size):
    """Resize a batch of numpy images to (size, size)."""
    from PIL import Image as PILImage
    out = []
    for img in imgs:
        pil = PILImage.fromarray(img.astype(np.uint8))
        pil = pil.resize((size, size), PILImage.LANCZOS)
        out.append(np.array(pil, dtype=np.float32))
    return np.stack(out)


CROP_SIZE = int(IMG_SIZE * 0.88)  # ~88% of image size
tta_preds = np.zeros((len(all_imgs), NUM_CLASSES))
n_views = 0

for flip in [False, True]:
    for crop_idx in range(5):
        batch_crops = []
        for img in all_imgs:
            img_use = img[:, ::-1, :] if flip else img  # horizontal flip
            crop    = five_crop(img_use, CROP_SIZE)[crop_idx]
            # Resize crop back to IMG_SIZE
            from PIL import Image as PILImage
            pil  = PILImage.fromarray(crop.astype(np.uint8))
            pil  = pil.resize((IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)
            batch_crops.append(np.array(pil, dtype=np.float32))

        batch = np.stack(batch_crops)
        tta_preds += model3.predict(batch, batch_size=BATCH_SIZE, verbose=0)
        n_views   += 1
        print(f"  View {n_views}/10 (flip={flip}, crop={crop_idx}) done")

tta_preds  /= n_views
pred_labels = np.argmax(tta_preds, axis=1)
tta_acc     = np.mean(pred_labels == true_labels)
print(f"\nTTA Accuracy (10-crop): {tta_acc*100:.2f}%")

## 16. Classification Report

Per-class precision/recall/F1 breakdown on the TTA predictions.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Classification Report                            ║
# ╚══════════════════════════════════════════════════════════════╝
emotion_names = [idx_to_name.get(i, str(i)) for i in range(NUM_CLASSES)]
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=emotion_names))

## 17. Confusion Matrix

Raw-count and row-normalized confusion matrices visualizing where the model confuses
emotions (e.g. Fear vs Sad).

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 17 — Confusion Matrix                                 ║
# ╚══════════════════════════════════════════════════════════════╝
cm      = confusion_matrix(true_labels, pred_labels)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names, yticklabels=emotion_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=emotion_names, yticklabels=emotion_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.suptitle(f'FER2013 — TTA Accuracy: {tta_acc*100:.2f}%', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → confusion_matrix.png")

## 18. Per-Class Accuracy Chart

Bar chart of accuracy per emotion class, compared against the paper's average
accuracy.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Per-class Bar Chart                              ║
# ╚══════════════════════════════════════════════════════════════╝
per_class = cm_norm.diagonal()
colors    = plt.cm.RdYlGn(per_class)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(emotion_names, per_class, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Per-class accuracy')
ax.axhline(0.6568, linestyle='--', color='orange', alpha=0.7, label='Paper DCNN avg')
ax.legend()
for bar, acc in zip(bars, per_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.2f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → per_class_accuracy.png")

## 19. Final Summary

Prints a side-by-side comparison of the paper baseline, the previous under-trained
attempt, and this version's raw/TTA test accuracy, plus the full list of changes made
and saves the final model.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Final Summary                                    ║
# ╚══════════════════════════════════════════════════════════════╝
improvement = (tta_acc - 0.6568) * 100
sign = '+' if improvement >= 0 else ''

print("\n" + "=" * 60)
print("  FINAL RESULTS")
print("=" * 60)
print(f"  Paper DCNN baseline      :  65.68%")
print(f"  Previous version         :  63.90%  (under-trained)")
print(f"  This version test acc    :  {test_acc*100:.2f}%")
print(f"  This version TTA acc     :  {tta_acc*100:.2f}%")
print(f"  Improvement over paper   :  {sign}{improvement:.2f}%")
print("=" * 60)

print("\n  CHANGES vs PREVIOUS VERSION:")
print("  1. EfficientNetV2M instead of V2S (2.5x more params)")
print("  2. Fixed LR schedule: flat-then-cosine (was pure cosine)")
print("  3. MixUp augmentation (alpha=0.3)")
print("  4. Focal Loss instead of label-smoothed CE")
print("  5. AdamW optimizer with weight_decay=1e-4")
print("  6. Load BEST P2 checkpoint for Phase 3 (not end weights)")
print("  7. 10-crop TTA (5-crop × flip) vs simple random augment TTA")
print("  8. Higher Phase 2 LR (2e-4 vs 5e-5) for faster convergence")
print("=" * 60)

print(f"\nAll outputs saved to: {OUT_DIR}/")
model3.save(f'{OUT_DIR}/fer_final_model.keras')
print("  fer_final_model.keras")
print("\nDone!")